In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("hw1data.tab", sep="\t")

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["Y"].value_counts(normalize=True))

print("\nMissing values:")
print(df.isnull().sum())

print("\nSMILES length statistics:")
print(df["Drug"].str.len().describe())

display(df.head())

Dataset shape: (10000, 3)

Class distribution:
Y
1    0.5236
0    0.4764
Name: proportion, dtype: float64

Missing values:
Drug_ID    0
Drug       0
Y          0
dtype: int64

SMILES length statistics:
count    10000.000000
mean        55.753600
std         13.489429
min         12.000000
25%         47.000000
50%         55.000000
75%         63.000000
max        177.000000
Name: Drug, dtype: float64


,Drug_ID,Drug,Y
0,0,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,1
1,1,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,0
2,2,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,0
3,3,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,0
4,4,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,0


In [6]:
from rdkit import Chem
import pandas as pd
import numpy as np

# Convert each SMILES string to an RDKit molecule
molecules = [
    Chem.MolFromSmiles(smi)
    for smi in df["Drug"]
]

# Check validity
valid_mask = np.array([
    mol is not None for mol in molecules
])

print("Total molecules:", len(df))
print("Valid SMILES:", valid_mask.sum())
print("Invalid SMILES:", (~valid_mask).sum())

# Convert valid molecules to canonical SMILES
canonical_smiles = [
    Chem.MolToSmiles(mol, canonical=True)
    if mol is not None else None
    for mol in molecules
]

valid_canonical = pd.Series(canonical_smiles).dropna()

print(
    "Duplicate canonical SMILES:",
    valid_canonical.duplicated().sum()
)

Total molecules: 10000
Valid SMILES: 10000
Invalid SMILES: 0
Duplicate canonical SMILES: 0


In [7]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import numpy as np

def morgan_features(smiles_list, radius=2, n_bits=2048):
    features = np.zeros(
        (len(smiles_list), n_bits),
        dtype=np.float32
    )

    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)

        if mol is None:
            raise ValueError(f"Invalid SMILES at index {i}: {smi}")

        fingerprint = AllChem.GetMorganGenerator(
            radius=radius,
            fpSize=n_bits
        ).GetFingerprint(mol)

        DataStructs.ConvertToNumpyArray(
            fingerprint,
            features[i]
        )

    return features

X_morgan = morgan_features(df["Drug"].tolist())

print("Shape:", X_morgan.shape)
print("Data type:", X_morgan.dtype)
print("First molecule active bits:", int(X_morgan[0].sum()))

Shape: (10000, 2048)
Data type: float32
First molecule active bits: 42


In [8]:
# Number of active bits per molecule
active_bits = X_morgan.sum(axis=1)

# Number of molecules activating each bit
bit_frequency = X_morgan.mean(axis=0)

print("Average active bits:", active_bits.mean())
print("Minimum active bits:", active_bits.min())
print("Maximum active bits:", active_bits.max())

print("Always zero bits:", np.sum(bit_frequency == 0))
print("Always one bits:", np.sum(bit_frequency == 1))

print("Variable bits:", np.sum(
    (bit_frequency > 0) & (bit_frequency < 1)
))

Average active bits: 55.7532
Minimum active bits: 4.0
Maximum active bits: 87.0
Always zero bits: 0
Always one bits: 0
Variable bits: 2048


In [9]:
from rdkit.Chem import Descriptors, rdMolDescriptors
import numpy as np
import pandas as pd

def molecular_descriptors(molecules):
    rows = []

    for mol in molecules:
        if mol is None:
            raise ValueError("Invalid molecule")

        rows.append([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            rdMolDescriptors.CalcTPSA(mol),
            rdMolDescriptors.CalcNumHBD(mol),
            rdMolDescriptors.CalcNumHBA(mol),
            rdMolDescriptors.CalcNumRotatableBonds(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcFractionCSP3(mol),
            sum(atom.GetFormalCharge() for atom in mol.GetAtoms()),
            sum(atom.GetAtomicNum() == 7 for atom in mol.GetAtoms())
        ])

    return np.asarray(rows, dtype=np.float32)

descriptor_names = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "AromaticRings",
    "FractionCSP3",
    "FormalCharge",
    "NitrogenCount"
]

X_desc = molecular_descriptors(molecules)

desc_df = pd.DataFrame(X_desc, columns=descriptor_names)

print("Descriptor matrix:", X_desc.shape)
print("Data type:", X_desc.dtype)

display(desc_df.describe().round(2))

Descriptor matrix: (10000, 10)
Data type: float32


,MolWt,MolLogP,TPSA,HBD,HBA,RotatableBonds,AromaticRings,FractionCSP3,FormalCharge,NitrogenCount
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,435.83,3.80,74.02,1.30,5.31,5.69,2.69,0.39,0.04,3.90
std,86.25,1.43,31.06,1.07,2.10,2.42,1.00,0.15,0.21,1.77
min,74.15,-4.61,0.00,0.00,0.00,0.00,0.00,0.00,-1.00,0.00
25%,375.81,2.87,52.88,1.00,4.00,4.00,2.00,0.30,0.00,3.00
50%,437.51,3.78,73.39,1.00,5.00,6.00,3.00,0.39,0.00,4.00
75%,492.45,4.67,94.81,2.00,7.00,7.00,3.00,0.48,0.00,5.00
max,794.48,10.29,207.28,8.00,14.00,22.00,7.00,1.00,3.00,11.00


In [10]:
# Add the hERG labels to the descriptor table
analysis_df = desc_df.copy()
analysis_df["hERG_Blocker"] = df["Y"].values

# Compare the two groups
group_means = analysis_df.groupby("hERG_Blocker").mean()

comparison = pd.DataFrame({
    "Non-blockers": group_means.loc[0],
    "Blockers": group_means.loc[1]
})

comparison["Difference"] = (
    comparison["Blockers"] -
    comparison["Non-blockers"]
)

display(comparison.round(3))

,Non-blockers,Blockers,Difference
MolWt,433.635010,437.833008,4.198000
MolLogP,3.322000,4.241000,0.919000
TPSA,83.322998,65.564003,-17.759001
HBD,1.524000,1.104000,-0.420000
HBA,5.641000,5.009000,-0.632000
RotatableBonds,5.459000,5.893000,0.434000
AromaticRings,2.559000,2.806000,0.247000
FractionCSP3,0.397000,0.389000,-0.008000
FormalCharge,0.026000,0.050000,0.023000
NitrogenCount,4.049000,3.756000,-0.293000


In [11]:
features_to_compare = [
    "MolLogP",
    "MolWt",
    "TPSA",
    "AromaticRings"
]

summary = (
    analysis_df
    .groupby("hERG_Blocker")[features_to_compare]
    .agg(["mean", "std", "median", "min", "max"])
)

display(summary.round(2))

MolLogP                                 MolWt             \
                mean   std median   min    max        mean        std   
hERG_Blocker                                                            
0               3.32  1.38   3.25 -1.94   9.33  433.640015  88.540001   
1               4.24  1.34   4.21 -4.61  10.29  437.829987  84.070000   

                                                       TPSA             \
                  median         min         max       mean        std   
hERG_Blocker                                                             
0             437.519989   74.150002  794.479980  83.320000  29.580000   
1             437.500000  180.270004  724.320007  65.559998  29.940001   

                                         AromaticRings                         
                 median  min         max          mean   std median  min  max  
hERG_Blocker                                                                   
0             83.900002  0.0  207.279999          2.56  1.01    3.0  0.0  7.0  
1             64.540001  0.0  206.600006          2.81  0.97    3.0  0.0  7.0

In [12]:
from scipy.stats import pointbiserialr
import pandas as pd
import numpy as np

rows = []

for feature in descriptor_names:
    group0 = analysis_df.loc[
        analysis_df["hERG_Blocker"] == 0, feature
    ]
    group1 = analysis_df.loc[
        analysis_df["hERG_Blocker"] == 1, feature
    ]

    pooled_std = np.sqrt(
        ((len(group0)-1)*group0.var(ddof=1)
         + (len(group1)-1)*group1.var(ddof=1))
        / (len(group0)+len(group1)-2)
    )

    cohen_d = (
        (group1.mean()-group0.mean()) / pooled_std
        if pooled_std > 0 else np.nan
    )

    correlation, _ = pointbiserialr(
        analysis_df["hERG_Blocker"],
        analysis_df[feature]
    )

    rows.append({
        "Descriptor": feature,
        "Cohens_d": cohen_d,
        "PointBiserial_r": correlation
    })

effect_df = pd.DataFrame(rows)
effect_df["Abs_d"] = effect_df["Cohens_d"].abs()

display(
    effect_df.sort_values("Abs_d", ascending=False)
    .drop(columns="Abs_d")
    .round(3)
)

,Descriptor,Cohens_d,PointBiserial_r
1,MolLogP,0.677,0.320
2,TPSA,-0.597,-0.286
3,HBD,-0.401,-0.196
4,HBA,-0.305,-0.150
6,AromaticRings,0.250,0.124
5,RotatableBonds,0.181,0.090
9,NitrogenCount,-0.166,-0.083
8,FormalCharge,0.109,0.054
7,FractionCSP3,-0.050,-0.025
0,MolWt,0.049,0.024


In [13]:
from rdkit.Chem.Scaffolds import MurckoScaffold

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    return Chem.MolToSmiles(scaffold, canonical=True)

df["Scaffold"] = df["Drug"].apply(get_scaffold)

print("Total molecules:", len(df))
print("Unique scaffolds:", df["Scaffold"].nunique())

display(df[["Drug", "Scaffold", "Y"]].head())

Total molecules: 10000
Unique scaffolds: 5185


,Drug,Scaffold,Y
0,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,c1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1,1
1,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,O=C1c2nc(Sc3ccccc3)ccc2CN1c1ccc(OCCN2CCCC2)cc1,0
2,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,O=C(CNc1n[nH]c2ccccc12)NC1CN(C2CCCCC2)C1,0
3,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,O=C1COc2ccc(CNC3CCC(CCn4c(=O)cnc5ccccc54)CC3)n...,0
4,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,O=C(NC1COc2cccc(-c3ccncc3)c2C1)c1cccnc1,0


In [14]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# 1. Create an array of molecule indices
indices = np.arange(len(df))

# 2. First split: Train+Validation vs Test
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

trainval_pos, test_pos = next(
    split1.split(
        df,
        groups=df["Scaffold"]
    )
)

# 3. Second split: Train vs Validation
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.125,
    random_state=42
)

train_pos, val_pos = next(
    split2.split(
        df.iloc[trainval_pos],
        groups=df.iloc[trainval_pos]["Scaffold"]
    )
)

# 4. Convert relative positions into original indices
train_idx = trainval_pos[train_pos]
val_idx = trainval_pos[val_pos]
test_idx = test_pos

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Train: 6841
Validation: 1074
Test: 2085


In [15]:
train_scaffolds = set(df.iloc[train_idx]["Scaffold"])
val_scaffolds = set(df.iloc[val_idx]["Scaffold"])
test_scaffolds = set(df.iloc[test_idx]["Scaffold"])

print("Train / Validation overlap:",
      len(train_scaffolds & val_scaffolds))

print("Train / Test overlap:",
      len(train_scaffolds & test_scaffolds))

print("Validation / Test overlap:",
      len(val_scaffolds & test_scaffolds))

for name, idx in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx)
]:
    labels = df.iloc[idx]["Y"]

    print(
        name,
        "| Molecules:", len(idx),
        "| Scaffolds:", df.iloc[idx]["Scaffold"].nunique(),
        "| Positive rate:", round(labels.mean(), 3)
    )

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0
Train | Molecules: 6841 | Scaffolds: 3629 | Positive rate: 0.526
Validation | Molecules: 1074 | Scaffolds: 519 | Positive rate: 0.518
Test | Molecules: 2085 | Scaffolds: 1037 | Positive rate: 0.52


In [16]:
X_train = X_morgan[train_idx]
X_val = X_morgan[val_idx]

y = df["Y"].to_numpy()

y_train = y[train_idx]
y_val = y[val_idx]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (6841, 2048)
X_val: (1074, 2048)
y_train: (6841,)
y_val: (1074,)


In [17]:
from sklearn.linear_model import LogisticRegression

model_morgan = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

model_morgan.fit(X_train, y_train)

LogisticRegression(max_iter=2000, random_state=42)

In [18]:
model_morgan.fit(X_train, y_train)

LogisticRegression(max_iter=2000, random_state=42)

In [19]:
prob_morgan = model_morgan.predict_proba(X_val)[:, 1]

print("First 10 predicted probabilities:")
print(prob_morgan[:10])

First 10 predicted probabilities:
[0.96115574 0.9147604  0.76854591 0.83654733 0.9902727  0.51474312
 0.38881983 0.08360562 0.87745414 0.97998542]


In [20]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

pred_morgan = (prob_morgan >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    pred_morgan,
    labels=[0, 1]
).ravel()

print("AUROC:", round(roc_auc_score(y_val, prob_morgan), 3))
print("AUPRC:", round(average_precision_score(y_val, prob_morgan), 3))
print("Sensitivity:", round(tp / (tp + fn), 3))
print("Specificity:", round(tn / (tn + fp), 3))
print("Precision:", round(tp / (tp + fp), 3))

AUROC: 0.791
AUPRC: 0.791
Sensitivity: 0.752
Specificity: 0.681
Precision: 0.717


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Select descriptor features using the SAME scaffold split
X_train_desc = X_desc[train_idx]
X_val_desc = X_desc[val_idx]

# Build a preprocessing + classification pipeline
model_desc = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

# Learn from training data
model_desc.fit(X_train_desc, y_train)

# Predict probabilities on unseen validation scaffolds
prob_desc = model_desc.predict_proba(X_val_desc)[:, 1]

# Convert probabilities into binary predictions
pred_desc = (prob_desc >= 0.5).astype(int)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(
    y_val, pred_desc, labels=[0, 1]
).ravel()

print("AUROC:", round(roc_auc_score(y_val, prob_desc), 3))
print("AUPRC:", round(average_precision_score(y_val, prob_desc), 3))
print("Sensitivity:", round(tp / (tp + fn), 3))
print("Specificity:", round(tn / (tn + fp), 3))
print("Precision:", round(tp / (tp + fp), 3))

AUROC: 0.743
AUPRC: 0.736
Sensitivity: 0.703
Specificity: 0.676
Precision: 0.699


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# 1. Combine both representations
X_combined = np.concatenate(
    [X_morgan, X_desc],
    axis=1
).astype(np.float32)

# 2. Use the same scaffold-based split
X_train_comb = X_combined[train_idx]
X_val_comb = X_combined[val_idx]

# 3. Define preprocessing
preprocessor = ColumnTransformer([
    (
        "descriptor_scaler",
        StandardScaler(),
        list(range(2048, 2058))
    )
], remainder="passthrough")

# 4. Build the model
model_comb = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

# 5. Train
model_comb.fit(X_train_comb, y_train)

# 6. Predict
prob_comb = model_comb.predict_proba(X_val_comb)[:, 1]
pred_comb = (prob_comb >= 0.5).astype(int)

# 7. Evaluate
tn, fp, fn, tp = confusion_matrix(
    y_val, pred_comb, labels=[0, 1]
).ravel()

print("AUROC:", round(roc_auc_score(y_val, prob_comb), 3))
print("AUPRC:", round(average_precision_score(y_val, prob_comb), 3))
print("Sensitivity:", round(tp / (tp + fn), 3))
print("Specificity:", round(tn / (tn + fp), 3))
print("Precision:", round(tp / (tp + fp), 3))

AUROC: 0.809
AUPRC: 0.803
Sensitivity: 0.781
Specificity: 0.72
Precision: 0.75


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

C_values = [0.001, 0.01, 0.1, 1, 10]

l1_results = []
l1_models = {}

for C in C_values:

    model = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("classifier", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C,
            max_iter=3000,
            random_state=42
        ))
    ])

    model.fit(X_train_comb, y_train)

    probabilities = model.predict_proba(
        X_val_comb
    )[:, 1]

    coefficients = model.named_steps[
        "classifier"
    ].coef_[0]

    n_selected = np.count_nonzero(
        np.abs(coefficients) > 1e-8
    )

    auc = roc_auc_score(y_val, probabilities)

    l1_results.append({
        "C": C,
        "AUROC": auc,
        "Selected_Features": n_selected,
        "Zero_Features": len(coefficients) - n_selected
    })

    l1_models[C] = model

l1_df = pd.DataFrame(l1_results)

display(l1_df.round(3))

,C,AUROC,Selected_Features,Zero_Features
0,0.001,0.733,1,2057
1,0.010,0.763,12,2046
2,0.100,0.821,225,1833
3,1.000,0.813,1253,805
4,10.000,0.746,1846,212


In [24]:
best_l1 = l1_models[0.1]

coefficients = best_l1.named_steps[
    "classifier"
].coef_[0]

# ColumnTransformer puts scaled descriptors first
descriptor_coefs = coefficients[:10]
morgan_coefs = coefficients[10:]

print(
    "Selected descriptors:",
    np.count_nonzero(np.abs(descriptor_coefs) > 1e-8)
)

print(
    "Selected Morgan bits:",
    np.count_nonzero(np.abs(morgan_coefs) > 1e-8)
)

descriptor_importance = pd.DataFrame({
    "Descriptor": descriptor_names,
    "Coefficient": descriptor_coefs
})

descriptor_importance["Absolute_Coefficient"] = (
    descriptor_importance["Coefficient"].abs()
)

display(
    descriptor_importance.sort_values(
        "Absolute_Coefficient",
        ascending=False
    ).round(4)
)

Selected descriptors: 7
Selected Morgan bits: 218


,Descriptor,Coefficient,Absolute_Coefficient
2,TPSA,-0.7726,0.7726
1,MolLogP,0.5384,0.5384
9,NitrogenCount,0.2812,0.2812
5,RotatableBonds,0.2775,0.2775
8,FormalCharge,0.1565,0.1565
6,AromaticRings,0.1482,0.1482
3,HBD,0.0244,0.0244
0,MolWt,0.0000,0.0000
4,HBA,0.0000,0.0000
7,FractionCSP3,0.0000,0.0000


In [25]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

groups_train = df.iloc[train_idx]["Scaffold"].to_numpy()

cv = GroupKFold(n_splits=5)

C_values = [0.01, 0.03, 0.1, 0.3, 1.0]

cv_results = []

for C in C_values:
    fold_scores = []

    for fold_train, fold_val in cv.split(
        X_train_comb,
        y_train,
        groups=groups_train
    ):
        model = Pipeline([
            ("preprocessing", clone(preprocessor)),
            ("classifier", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C,
                max_iter=3000,
                random_state=42
            ))
        ])

        model.fit(
            X_train_comb[fold_train],
            y_train[fold_train]
        )

        probabilities = model.predict_proba(
            X_train_comb[fold_val]
        )[:, 1]

        auc = roc_auc_score(
            y_train[fold_val],
            probabilities
        )

        fold_scores.append(auc)

    cv_results.append({
        "C": C,
        "Mean_AUROC": np.mean(fold_scores),
        "Std_AUROC": np.std(fold_scores),
    })

cv_df = pd.DataFrame(cv_results)
display(cv_df.round(4))

,C,Mean_AUROC,Std_AUROC
0,0.01,0.7335,0.0195
1,0.03,0.7501,0.0223
2,0.10,0.7936,0.0198
3,0.30,0.8135,0.0215
4,1.00,0.8088,0.0183


In [26]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

model_l1_cv = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("classifier", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.3,
        max_iter=3000,
        random_state=42
    ))
])

model_l1_cv.fit(X_train_comb, y_train)

prob_l1_cv = model_l1_cv.predict_proba(X_val_comb)[:, 1]

print("Validation AUROC:",
      round(roc_auc_score(y_val, prob_l1_cv), 4))

print("Validation AUPRC:",
      round(average_precision_score(y_val, prob_l1_cv), 4))

coefs = model_l1_cv.named_steps["classifier"].coef_[0]

print("Selected features:",
      np.count_nonzero(np.abs(coefs) > 1e-8))

Validation AUROC: 0.8345
Validation AUPRC: 0.839
Selected features: 659


In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import time

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

start = time.perf_counter()

rf_model.fit(X_train_comb, y_train)

training_time = time.perf_counter() - start

prob_rf = rf_model.predict_proba(X_val_comb)[:, 1]

print("Validation AUROC:",
      round(roc_auc_score(y_val, prob_rf), 4))

print("Validation AUPRC:",
      round(average_precision_score(y_val, prob_rf), 4))

print("Training time (seconds):",
      round(training_time, 2))

Validation AUROC: 0.8761
Validation AUPRC: 0.8898
Training time (seconds): 0.38


In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
import pandas as pd
import numpy as np

def evaluate_model(y_true, probabilities, model_name,
                   threshold=0.5):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    results = {
        "Model": model_name,
        "Threshold": threshold,
        "AUROC": roc_auc_score(y_true, probabilities),
        "AUPRC": average_precision_score(
            y_true, probabilities
        ),
        "Accuracy": accuracy_score(
            y_true, predictions
        ),
        "Sensitivity": recall_score(
            y_true, predictions
        ),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        )
    }

    return results

In [29]:
all_results = [
    evaluate_model(
        y_val, prob_morgan,
        "Morgan + LR"
    ),

    evaluate_model(
        y_val, prob_desc,
        "Descriptors + LR"
    ),

    evaluate_model(
        y_val, prob_comb,
        "Combined + LR (L2)"
    ),

    evaluate_model(
        y_val, prob_l1_cv,
        "Combined + LR (L1)"
    ),

    evaluate_model(
        y_val, prob_rf,
        "Combined + RF"
    )
]

comparison_df = pd.DataFrame(all_results)

display(comparison_df.round(4))

,Model,Threshold,AUROC,AUPRC,Accuracy,Sensitivity,Specificity,Precision,F1
0,Morgan + LR,0.5,0.7913,0.7907,0.7179,0.7518,0.6815,0.7170,0.7340
1,Descriptors + LR,0.5,0.7431,0.7357,0.6899,0.7032,0.6757,0.6995,0.7013
2,Combined + LR (L2),0.5,0.8091,0.8026,0.7514,0.7806,0.7201,0.7496,0.7648
3,Combined + LR (L1),0.5,0.8345,0.8390,0.7486,0.7662,0.7297,0.7527,0.7594
4,Combined + RF,0.5,0.8761,0.8898,0.7849,0.7914,0.7780,0.7928,0.7921


In [30]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

models_cv = {
    "Morgan + LR": LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ),

    "Descriptors + LR": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Combined + LR (L2)": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("classifier", LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Combined + LR (L1)": Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("classifier", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.3,
            max_iter=3000,
            random_state=42
        ))
    ]),

    "Combined + RF": RandomForestClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=2
    )
}

In [31]:
feature_sets_cv = {
    "Morgan + LR": X_morgan,
    "Descriptors + LR": X_desc,
    "Combined + LR (L2)": X_combined,
    "Combined + LR (L1)": X_combined,
    "Combined + RF": X_combined
}

In [32]:
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd
import time

groups_train = df.iloc[train_idx]["Scaffold"].to_numpy()

cv = GroupKFold(n_splits=5)

all_cv_results = []

for model_name, model_template in models_cv.items():

    X = feature_sets_cv[model_name][train_idx]

    for fold_number, (tr, va) in enumerate(
        cv.split(X, y_train, groups=groups_train),
        start=1
    ):

        model = clone(model_template)

        start_time = time.perf_counter()

        model.fit(X[tr], y_train[tr])

        training_time = time.perf_counter() - start_time

        start_time = time.perf_counter()

        probabilities = model.predict_proba(X[va])[:, 1]

        inference_time = time.perf_counter() - start_time

        metrics = evaluate_model(
            y_true=y_train[va],
            probabilities=probabilities,
            model_name=model_name,
            threshold=0.5
        )

        metrics["Fold"] = fold_number
        metrics["Train_Time_s"] = training_time
        metrics["Inference_Time_s"] = inference_time
        metrics["Inference_ms_per_molecule"] = (
            inference_time * 1000 / len(va)
        )

        all_cv_results.append(metrics)

cv_full_df = pd.DataFrame(all_cv_results)

display(cv_full_df.round(4))

,Model,Threshold,AUROC,AUPRC,Accuracy,Sensitivity,Specificity,Precision,F1,Fold,Train_Time_s,Inference_Time_s,Inference_ms_per_molecule
0,Morgan + LR,0.5,0.8201,0.8449,0.7546,0.7683,0.7391,0.7683,0.7683,1,0.0990,0.0030,0.0022
1,Morgan + LR,0.5,0.7989,0.8313,0.7200,0.7484,0.6816,0.7610,0.7546,2,0.1320,0.0027,0.0020
2,Morgan + LR,0.5,0.8077,0.8060,0.7346,0.7525,0.7160,0.7346,0.7435,3,0.1306,0.0029,0.0021
3,Morgan + LR,0.5,0.7568,0.7488,0.6893,0.7093,0.6691,0.6844,0.6966,4,0.1395,0.0028,0.0020
4,Morgan + LR,0.5,0.7919,0.7871,0.7230,0.7202,0.7258,0.7318,0.7260,5,0.1146,0.0027,0.0020
5,Descriptors + LR,0.5,0.7624,0.7874,0.6771,0.7379,0.6087,0.6798,0.7077,1,0.0070,0.0005,0.0003
6,Descriptors + LR,0.5,0.7249,0.7702,0.6520,0.6595,0.6420,0.7139,0.6856,2,0.0067,0.0004,0.0003
7,Descriptors + LR,0.5,0.7047,0.7225,0.6418,0.6667,0.6158,0.6445,0.6554,3,0.0075,0.0004,0.0003
8,Descriptors + LR,0.5,0.7228,0.6875,0.6791,0.7006,0.6574,0.6741,0.6871,4,0.0072,0.0004,0.0003
9,Descriptors + LR,0.5,0.7620,0.7832,0.6740,0.7403,0.6051,0.6607,0.6982,5,0.0072,0.0004,0.0003


In [33]:
metrics_to_compare = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Train_Time_s",
    "Inference_ms_per_molecule"
]

cv_summary = cv_full_df.groupby("Model")[
    metrics_to_compare
].agg(["mean", "std"])

display(cv_summary.round(4))

AUROC           AUPRC         Accuracy          \
                      mean     std    mean     std     mean     std   
Model                                                                 
Combined + LR (L1)  0.8135  0.0241  0.8237  0.0400   0.7440  0.0268   
Combined + LR (L2)  0.8083  0.0220  0.8142  0.0353   0.7401  0.0213   
Combined + RF       0.8609  0.0308  0.8719  0.0413   0.7822  0.0299   
Descriptors + LR    0.7354  0.0257  0.7502  0.0435   0.6648  0.0168   
Morgan + LR         0.7951  0.0239  0.8036  0.0379   0.7243  0.0238   

                   Sensitivity         Specificity         Precision          \
                          mean     std        mean     std      mean     std   
Model                                                                          
Combined + LR (L1)      0.7555  0.0345      0.7294  0.0457    0.7565  0.0340   
Combined + LR (L2)      0.7556  0.0231      0.7214  0.0378    0.7507  0.0324   
Combined + RF           0.7910  0.0237      0.7721  0.0426    0.7930  0.0457   
Descriptors + LR        0.7010  0.0381      0.6258  0.0228    0.6746  0.0258   
Morgan + LR             0.7397  0.0243      0.7063  0.0298    0.7360  0.0330   

                        F1         Train_Time_s          \
                      mean     std         mean     std   
Model                                                     
Combined + LR (L1)  0.7557  0.0302       0.2496  0.0822   
Combined + LR (L2)  0.7530  0.0256       0.2184  0.0324   
Combined + RF       0.7918  0.0335       1.7505  0.3295   
Descriptors + LR    0.6868  0.0197       0.0071  0.0003   
Morgan + LR         0.7378  0.0277       0.1231  0.0163   

                   Inference_ms_per_molecule          
                                        mean     std  
Model                                                 
Combined + LR (L1)                    0.0148  0.0020  
Combined + LR (L2)                    0.0147  0.0010  
Combined + RF                         0.0333  0.0083  
Descriptors + LR                      0.0003  0.0000  
Morgan + LR                           0.0021  0.0001

In [34]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

def threshold_search(y_true, probabilities):

    # Try all distinct predicted probabilities as thresholds
    thresholds = np.unique(
        np.concatenate(([0.0], probabilities, [1.0]))
    )

    rows = []

    for t in thresholds:
        pred = (probabilities >= t).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, pred, labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if tp + fn else 0
        specificity = tn / (tn + fp) if tn + fp else 0
        precision = tp / (tp + fp) if tp + fp else 0

        rows.append({
            "Threshold": t,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision
        })

    return pd.DataFrame(rows)

In [35]:
def choose_thresholds(y_true, probabilities):

    table = threshold_search(y_true, probabilities)

    goals = {
        "Sensitivity": (
            "Sensitivity", "Specificity"
        ),
        "Specificity": (
            "Specificity", "Sensitivity"
        ),
        "Precision": (
            "Precision", "Sensitivity"
        )
    }

    selected = {}

    for goal, (maximize, constraint) in goals.items():

        eligible = table[
            table[constraint] > 0.70
        ]

        if eligible.empty:
            selected[goal] = None
        else:
            best = eligible.sort_values(
                [maximize, constraint],
                ascending=[False, False]
            ).iloc[0]

            selected[goal] = float(best["Threshold"])

    return selected

In [36]:
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 15],
    "min_samples_leaf": [1, 3]
}

outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=3)

X = X_combined[train_idx]
y_cv = y_train
groups = groups_train

nested_results = []

for outer_fold, (outer_tr, outer_te) in enumerate(
    outer_cv.split(X, y_cv, groups), start=1
):

    X_outer_train = X[outer_tr]
    y_outer_train = y_cv[outer_tr]
    groups_outer = groups[outer_tr]

    X_outer_test = X[outer_te]
    y_outer_test = y_cv[outer_te]

    candidates = []

    # INNER LOOP
    for params in ParameterGrid(param_grid):

        oof_prob = np.zeros(len(y_outer_train))

        for inner_tr, inner_val in inner_cv.split(
            X_outer_train,
            y_outer_train,
            groups_outer
        ):

            model = RandomForestClassifier(
                **params,
                random_state=42,
                max_features="sqrt",
                n_jobs=2
            )

            model.fit(
                X_outer_train[inner_tr],
                y_outer_train[inner_tr]
            )

            oof_prob[inner_val] = (
                model.predict_proba(
                    X_outer_train[inner_val]
                )[:, 1]
            )

        auc = roc_auc_score(y_outer_train, oof_prob)

        thresholds = choose_thresholds(
            y_outer_train, oof_prob
        )

        candidates.append({
            "params": params,
            "AUROC": auc,
            "thresholds": thresholds,
            "oof_prob": oof_prob
        })

    # Select best hyperparameters by inner AUROC
    best = max(candidates, key=lambda x: x["AUROC"])

    # Refit on entire outer training fold
    final_model = RandomForestClassifier(
        **best["params"],
        random_state=42,
        max_features="sqrt",
        n_jobs=2
    )

    final_model.fit(X_outer_train, y_outer_train)

    outer_prob = final_model.predict_proba(
        X_outer_test
    )[:, 1]

    # Evaluate outer AUROC
    result = {
        "Outer_Fold": outer_fold,
        "AUROC": roc_auc_score(
            y_outer_test, outer_prob
        ),
        "Inner_Best_Params": best["params"]
    }

    # Evaluate each inner-selected threshold
    for goal, threshold in best["thresholds"].items():

        if threshold is not None:
            metrics = evaluate_model(
                y_outer_test,
                outer_prob,
                "Random Forest",
                threshold=threshold
            )

            result[f"{goal}_Threshold"] = threshold
            result[f"{goal}_Sensitivity"] = metrics["Sensitivity"]
            result[f"{goal}_Specificity"] = metrics["Specificity"]
            result[f"{goal}_Precision"] = metrics["Precision"]

    nested_results.append(result)

nested_df = pd.DataFrame(nested_results)
display(nested_df)


,Outer_Fold,AUROC,Inner_Best_Params,Sensitivity_Threshold,Sensitivity_Sensitivity,Sensitivity_Specificity,Sensitivity_Precision,Specificity_Threshold,Specificity_Sensitivity,Specificity_Specificity,Specificity_Precision,Precision_Threshold,Precision_Sensitivity,Precision_Specificity,Precision_Precision
0,1,0.891137,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.481250,0.834483,0.774845,0.806667,0.558333,0.755862,0.854037,0.853583,0.558333,0.755862,0.854037,0.853583
1,2,0.896480,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.456000,0.871665,0.736661,0.817640,0.541500,0.804320,0.827883,0.863574,0.541500,0.804320,0.827883,0.863574
2,3,0.849588,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.478333,0.805436,0.714499,0.746684,0.565000,0.701001,0.816143,0.799347,0.565000,0.701001,0.816143,0.799347
3,4,0.825516,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.457500,0.813953,0.680882,0.720721,0.579667,0.664244,0.817647,0.786575,0.579667,0.664244,0.817647,0.786575
4,5,0.881513,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.472500,0.813486,0.777943,0.791899,0.561250,0.704448,0.874814,0.853913,0.561250,0.704448,0.874814,0.853913


In [37]:
best = max(candidates, key=lambda x: x["AUROC"])

In [38]:
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

new_models = {
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=2
    ),

    "SVM-RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=42
        ))
    ]),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.1,
        max_leaf_nodes=31,
        random_state=42
    )
}

In [39]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import pandas as pd
import numpy as np
import time

cv = GroupKFold(n_splits=5)

X = X_combined[train_idx]
y_cv = y_train
groups = groups_train

new_cv_results = []

for model_name, model_template in new_models.items():

    print(f"Running: {model_name}")

    for fold_number, (tr, va) in enumerate(
        cv.split(X, y_cv, groups=groups),
        start=1
    ):

        model = clone(model_template)

        start = time.perf_counter()

        model.fit(X[tr], y_cv[tr])

        train_time = time.perf_counter() - start

        start = time.perf_counter()

        probabilities = model.predict_proba(X[va])[:, 1]

        inference_time = time.perf_counter() - start

        metrics = evaluate_model(
            y_true=y_cv[va],
            probabilities=probabilities,
            model_name=model_name,
            threshold=0.5
        )

        metrics["Fold"] = fold_number
        metrics["Train_Time_s"] = train_time
        metrics["Inference_ms_per_molecule"] = (
            inference_time * 1000 / len(va)
        )

        new_cv_results.append(metrics)

new_cv_df = pd.DataFrame(new_cv_results)

display(new_cv_df.round(4))

Running: Extra Trees
Running: HistGradientBoosting


,Model,Threshold,AUROC,AUPRC,Accuracy,Sensitivity,Specificity,Precision,F1,Fold,Train_Time_s,Inference_ms_per_molecule
0,Extra Trees,0.5,0.8830,0.8939,0.7991,0.8000,0.7981,0.8169,0.8084,1,2.6415,0.0330
1,Extra Trees,0.5,0.8759,0.9058,0.7975,0.8056,0.7866,0.8364,0.8207,2,2.8152,0.0331
2,Extra Trees,0.5,0.8449,0.8588,0.7580,0.7811,0.7339,0.7541,0.7674,3,2.6152,0.0330
3,Extra Trees,0.5,0.8109,0.8105,0.7325,0.7326,0.7324,0.7347,0.7336,4,2.6014,0.0331
4,Extra Trees,0.5,0.8640,0.8762,0.7836,0.7604,0.8077,0.8042,0.7817,5,2.6327,0.0328
5,SVM-RBF,0.5,0.8680,0.8767,0.7838,0.7945,0.7717,0.7967,0.7956,1,99.7434,4.6931
6,SVM-RBF,0.5,0.8490,0.8895,0.7617,0.7751,0.7435,0.8037,0.7891,2,100.2188,5.1488
7,SVM-RBF,0.5,0.8409,0.8509,0.7522,0.7539,0.7504,0.7594,0.7566,3,99.6522,5.5353
8,SVM-RBF,0.5,0.7818,0.7820,0.7142,0.7311,0.6971,0.7094,0.7201,4,99.1556,4.5433
9,SVM-RBF,0.5,0.8375,0.8486,0.7675,0.7389,0.7973,0.7911,0.7641,5,89.5273,4.2820


In [40]:
all_cv_df = pd.concat(
    [cv_full_df, new_cv_df],
    ignore_index=True
)

metrics_to_compare = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Train_Time_s",
    "Inference_ms_per_molecule"
]

all_cv_summary = all_cv_df.groupby("Model")[
    metrics_to_compare
].agg(["mean", "std"])

all_cv_summary = all_cv_summary.sort_values(
    ("AUROC", "mean"),
    ascending=False
)

display(all_cv_summary.round(4))

AUROC           AUPRC         Accuracy          \
                        mean     std    mean     std     mean     std   
Model                                                                   
Combined + RF         0.8609  0.0308  0.8719  0.0413   0.7822  0.0299   
Extra Trees           0.8557  0.0289  0.8690  0.0372   0.7742  0.0285   
HistGradientBoosting  0.8497  0.0270  0.8608  0.0380   0.7660  0.0287   
SVM-RBF               0.8354  0.0322  0.8495  0.0415   0.7559  0.0260   
Combined + LR (L1)    0.8135  0.0241  0.8237  0.0400   0.7440  0.0268   
Combined + LR (L2)    0.8083  0.0220  0.8142  0.0353   0.7401  0.0213   
Morgan + LR           0.7951  0.0239  0.8036  0.0379   0.7243  0.0238   
Descriptors + LR      0.7354  0.0257  0.7502  0.0435   0.6648  0.0168   

                     Sensitivity         Specificity         Precision  \
                            mean     std        mean     std      mean   
Model                                                                    
Combined + RF             0.7910  0.0237      0.7721  0.0426    0.7930   
Extra Trees               0.7759  0.0300      0.7717  0.0360    0.7893   
HistGradientBoosting      0.7692  0.0292      0.7617  0.0353    0.7806   
SVM-RBF                   0.7587  0.0261      0.7520  0.0372    0.7721   
Combined + LR (L1)        0.7555  0.0345      0.7294  0.0457    0.7565   
Combined + LR (L2)        0.7556  0.0231      0.7214  0.0378    0.7507   
Morgan + LR               0.7397  0.0243      0.7063  0.0298    0.7360   
Descriptors + LR          0.7010  0.0381      0.6258  0.0228    0.6746   

                                  F1         Train_Time_s          \
                         std    mean     std         mean     std   
Model                                                               
Combined + RF         0.0457  0.7918  0.0335       1.7505  0.3295   
Extra Trees           0.0431  0.7824  0.0344       2.6612  0.0875   
HistGradientBoosting  0.0423  0.7747  0.0347       3.7808  0.9788   
SVM-RBF               0.0389  0.7651  0.0300      97.6595  4.5617   
Combined + LR (L1)    0.0340  0.7557  0.0302       0.2496  0.0822   
Combined + LR (L2)    0.0324  0.7530  0.0256       0.2184  0.0324   
Morgan + LR           0.0330  0.7378  0.0277       0.1231  0.0163   
Descriptors + LR      0.0258  0.6868  0.0197       0.0071  0.0003   

                     Inference_ms_per_molecule          
                                          mean     std  
Model                                                   
Combined + RF                           0.0333  0.0083  
Extra Trees                             0.0330  0.0001  
HistGradientBoosting                    0.0171  0.0033  
SVM-RBF                                 4.8405  0.4999  
Combined + LR (L1)                      0.0148  0.0020  
Combined + LR (L2)                      0.0147  0.0010  
Morgan + LR                             0.0021  0.0001  
Descriptors + LR                        0.0003  0.0000

In [41]:
import numpy as np
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

def make_morgan_features(molecules, radius, n_bits):
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=n_bits
    )

    features = np.zeros(
        (len(molecules), n_bits),
        dtype=np.float32
    )

    for i, mol in enumerate(molecules):
        if mol is None:
            raise ValueError(f"Invalid molecule at row {i}")

        fp = generator.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, features[i])

    return features

In [42]:
fingerprint_settings = [
    (2, 1024),
    (2, 2048),
    (3, 1024),
    (3, 2048)
]

representation_sets = {}

for radius, n_bits in fingerprint_settings:

    X_fp = make_morgan_features(
        molecules,
        radius=radius,
        n_bits=n_bits
    )

    X_comb = np.concatenate(
        [X_fp, X_desc],
        axis=1
    ).astype(np.float32)

    name = f"R{radius}_B{n_bits}"
    representation_sets[name] = X_comb

    print(name, X_comb.shape)

R2_B1024 (10000, 1034)
R2_B2048 (10000, 2058)
R3_B1024 (10000, 1034)
R3_B2048 (10000, 2058)


In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import pandas as pd

rf_template = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

cv = GroupKFold(n_splits=5)

results = []

for name, X_all in representation_sets.items():

    print("Evaluating:", name)

    X = X_all[train_idx]

    for fold, (tr, va) in enumerate(
        cv.split(X, y_train, groups=groups_train),
        start=1
    ):

        model = clone(rf_template)

        model.fit(X[tr], y_train[tr])

        probabilities = model.predict_proba(
            X[va]
        )[:, 1]

        metrics = evaluate_model(
            y_true=y_train[va],
            probabilities=probabilities,
            model_name=name,
            threshold=0.5
        )

        metrics["Fold"] = fold
        results.append(metrics)

fingerprint_cv_df = pd.DataFrame(results)

Evaluating: R2_B1024
Evaluating: R2_B2048
Evaluating: R3_B1024
Evaluating: R3_B2048


In [44]:
metrics = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1"
]

fingerprint_summary = (
    fingerprint_cv_df
    .groupby("Model")[metrics]
    .agg(["mean", "std"])
    .sort_values(("AUROC", "mean"), ascending=False)
)

display(fingerprint_summary.round(4))

AUROC           AUPRC         Accuracy         Sensitivity          \
            mean     std    mean     std     mean     std        mean     std   
Model                                                                           
R2_B1024  0.8610  0.0287  0.8722  0.0384   0.7761  0.0272      0.7829  0.0312   
R2_B2048  0.8609  0.0308  0.8719  0.0413   0.7822  0.0299      0.7910  0.0237   
R3_B2048  0.8609  0.0285  0.8723  0.0362   0.7799  0.0241      0.7883  0.0272   
R3_B1024  0.8602  0.0287  0.8710  0.0378   0.7807  0.0286      0.7858  0.0310   

         Specificity         Precision              F1          
                mean     std      mean     std    mean     std  
Model                                                           
R2_B1024      0.7677  0.0301    0.7876  0.0400  0.7852  0.0342  
R2_B2048      0.7721  0.0426    0.7930  0.0457  0.7918  0.0335  
R3_B2048      0.7694  0.0318    0.7906  0.0367  0.7893  0.0302  
R3_B1024      0.7745  0.0291    0.7930  0.0401  0.7893  0.0350

In [45]:
for name in [
    "molecules",
    "X_desc",
    "train_idx",
    "y_train",
    "groups_train",
    "rf_template",
    "evaluate_model"
]:
    print(name, name in globals())

molecules True
X_desc True
train_idx True
y_train True
groups_train True
rf_template True
evaluate_model True


In [46]:
import numpy as np
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

def make_fingerprint_features(molecules, fp_type):

    feature_mode = fp_type.startswith("Feature")
    count_mode = fp_type.endswith("Count")

    if feature_mode:
        atom_inv = rdFingerprintGenerator.GetMorganFeatureAtomInvGen()

        generator = rdFingerprintGenerator.GetMorganGenerator(
            radius=2,
            fpSize=2048,
            atomInvariantsGenerator=atom_inv
        )
    else:
        generator = rdFingerprintGenerator.GetMorganGenerator(
            radius=2,
            fpSize=2048
        )

    X_fp = np.zeros(
        (len(molecules), 2048),
        dtype=np.float32
    )

    for i, mol in enumerate(molecules):

        if mol is None:
            raise ValueError(f"Invalid molecule at row {i}")

        if count_mode:
            fp = generator.GetCountFingerprint(mol)
        else:
            fp = generator.GetFingerprint(mol)

        DataStructs.ConvertToNumpyArray(fp, X_fp[i])

    return X_fp

In [47]:
fp_types = [
    "Standard_Binary",
    "Standard_Count",
    "Feature_Binary",
    "Feature_Count"
]

fp_type_sets = {}

for fp_type in fp_types:

    X_fp = make_fingerprint_features(
        molecules,
        fp_type
    )

    X_comb = np.concatenate(
        [X_fp, X_desc],
        axis=1
    ).astype(np.float32)

    fp_type_sets[fp_type] = X_comb

    print(fp_type, X_comb.shape)

Standard_Binary (10000, 2058)
Standard_Count (10000, 2058)
Feature_Binary (10000, 2058)
Feature_Count (10000, 2058)


In [48]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import pandas as pd

rf_template = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

cv = GroupKFold(n_splits=5)

fp_type_results = []

for name, X_all in fp_type_sets.items():

    print("Evaluating:", name)

    X = X_all[train_idx]

    for fold, (tr, va) in enumerate(
        cv.split(
            X,
            y_train,
            groups=groups_train
        ),
        start=1
    ):

        model = clone(rf_template)

        model.fit(
            X[tr],
            y_train[tr]
        )

        probabilities = model.predict_proba(
            X[va]
        )[:, 1]

        metrics = evaluate_model(
            y_true=y_train[va],
            probabilities=probabilities,
            model_name=name,
            threshold=0.5
        )

        metrics["Fold"] = fold

        fp_type_results.append(metrics)

fp_type_cv_df = pd.DataFrame(fp_type_results)

Evaluating: Standard_Binary
Evaluating: Standard_Count
Evaluating: Feature_Binary
Evaluating: Feature_Count


In [49]:
metrics = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1"
]

fp_type_summary = (
    fp_type_cv_df
    .groupby("Model")[metrics]
    .agg(["mean", "std"])
    .sort_values(
        ("AUROC", "mean"),
        ascending=False
    )
)

display(fp_type_summary.round(4))

AUROC           AUPRC         Accuracy         Sensitivity  \
                   mean     std    mean     std     mean     std        mean   
Model                                                                          
Feature_Count    0.8688  0.0282  0.8804  0.0370   0.7879  0.0307      0.7867   
Feature_Binary   0.8661  0.0285  0.8775  0.0380   0.7828  0.0333      0.7844   
Standard_Count   0.8655  0.0287  0.8758  0.0385   0.7860  0.0289      0.7956   
Standard_Binary  0.8609  0.0308  0.8719  0.0413   0.7822  0.0299      0.7910   

                        Specificity         Precision              F1          
                    std        mean     std      mean     std    mean     std  
Model                                                                          
Feature_Count    0.0274      0.7884  0.0405    0.8044  0.0427  0.7953  0.0340  
Feature_Binary   0.0285      0.7803  0.0438    0.7979  0.0462  0.7909  0.0363  
Standard_Count   0.0260      0.7746  0.0381    0.7958  0.0421  0.7955  0.0331  
Standard_Binary  0.0237      0.7721  0.0426    0.7930  0.0457  0.7918  0.0335

In [50]:
for name, X in fp_type_sets.items():
    print(
        name,
        "Max fingerprint value:",
        X[:, :2048].max()
    )

Standard_Binary Max fingerprint value: 1.0
Standard_Count Max fingerprint value: 26.0
Feature_Binary Max fingerprint value: 1.0
Feature_Count Max fingerprint value: 38.0


In [51]:
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

descriptor_functions = Descriptors._descList

extended_descriptor_names = [
    name for name, func in descriptor_functions
]

X_desc_extended = np.array([
    [func(mol) for name, func in descriptor_functions]
    for mol in molecules
], dtype=np.float64)

print("Descriptor matrix shape:", X_desc_extended.shape)

print(
    "Number of non-finite values:",
    np.sum(~np.isfinite(X_desc_extended))
)

print(
    "Number of constant descriptors:",
    np.sum(
        np.nanstd(X_desc_extended, axis=0) == 0
    )
)

Descriptor matrix shape: (10000, 210)
Number of non-finite values: 24
Number of constant descriptors: 9


In [52]:
desc_df = pd.DataFrame(
    X_desc_extended,
    columns=extended_descriptor_names
)

# Identify descriptors with NaN or infinity
nonfinite_counts = (
    (~np.isfinite(desc_df.to_numpy()))
    .sum(axis=0)
)

nonfinite_descriptors = pd.Series(
    nonfinite_counts,
    index=desc_df.columns
)

nonfinite_descriptors = nonfinite_descriptors[
    nonfinite_descriptors > 0
]

print("Descriptors with non-finite values:")
display(nonfinite_descriptors)

# Identify constant descriptors
constant_descriptors = []

for column in desc_df.columns:
    values = desc_df[column].to_numpy()
    finite_values = values[np.isfinite(values)]

    if len(finite_values) == 0:
        constant_descriptors.append(column)
    elif np.all(finite_values == finite_values[0]):
        constant_descriptors.append(column)

print("\nConstant descriptors:")
print(constant_descriptors)

Descriptors with non-finite values:


BCUT2D_MWHI       3
BCUT2D_MWLOW      3
BCUT2D_CHGHI      3
BCUT2D_CHGLO      3
BCUT2D_LOGPHI     3
BCUT2D_LOGPLOW    3
BCUT2D_MRHI       3
BCUT2D_MRLOW      3
dtype: int64


Constant descriptors:
['SMR_VSA8', 'SlogP_VSA9', 'fr_aldehyde', 'fr_azide', 'fr_diazo', 'fr_isocyan', 'fr_isothiocyan', 'fr_prisulfonamd', 'fr_thiocyan']


In [53]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
import numpy as np

extended_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("variance_filter", VarianceThreshold(threshold=0.0)),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=2
    ))
])

In [54]:
# Feature Morgan Count from our previous experiment
X_fp = fp_type_sets["Feature_Count"][:, :2048]

# Convert infinity to NaN
X_extended_clean = X_desc_extended.copy()

X_extended_clean[
    ~np.isfinite(X_extended_clean)
] = np.nan

# Combine fingerprint and extended descriptors
X_full_extended = np.concatenate(
    [X_fp, X_extended_clean],
    axis=1
).astype(np.float32)

print("Combined shape:", X_full_extended.shape)
print("Remaining NaN values:", np.isnan(X_full_extended).sum())
print("Infinite values:", np.isinf(X_full_extended).sum())

Combined shape: (10000, 2258)
Remaining NaN values: 24
Infinite values: 0


In [55]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import pandas as pd

cv = GroupKFold(n_splits=5)

extended_results = []

X = X_full_extended[train_idx]

for fold, (tr, va) in enumerate(
    cv.split(X, y_train, groups=groups_train),
    start=1
):

    model = clone(extended_rf)

    model.fit(X[tr], y_train[tr])

    probabilities = model.predict_proba(X[va])[:, 1]

    metrics = evaluate_model(
        y_true=y_train[va],
        probabilities=probabilities,
        model_name="Feature Count + Extended Descriptors",
        threshold=0.5
    )

    metrics["Fold"] = fold
    extended_results.append(metrics)

extended_cv_df = pd.DataFrame(extended_results)

In [56]:
baseline = fp_type_cv_df[
    fp_type_cv_df["Model"] == "Feature_Count"
].copy()

baseline["Model"] = "Feature Count + 10 Descriptors"

comparison = pd.concat(
    [baseline, extended_cv_df],
    ignore_index=True
)

descriptor_comparison = (
    comparison.groupby("Model")[
        ["AUROC", "AUPRC", "Accuracy",
         "Sensitivity", "Specificity",
         "Precision", "F1"]
    ]
    .agg(["mean", "std"])
)

display(descriptor_comparison.round(4))

AUROC           AUPRC         Accuracy  \
                                        mean     std    mean     std     mean   
Model                                                                           
Feature Count + 10 Descriptors        0.8688  0.0282  0.8804  0.0370   0.7879   
Feature Count + Extended Descriptors  0.8677  0.0293  0.8773  0.0372   0.7898   

                                             Sensitivity         Specificity  \
                                         std        mean     std        mean   
Model                                                                          
Feature Count + 10 Descriptors        0.0307      0.7867  0.0274      0.7884   
Feature Count + Extended Descriptors  0.0336      0.7863  0.0307      0.7931   

                                             Precision              F1          
                                         std      mean     std    mean     std  
Model                                                                           
Feature Count + 10 Descriptors        0.0405    0.8044  0.0427  0.7953  0.0340  
Feature Count + Extended Descriptors  0.0469    0.8078  0.0481  0.7966  0.0368

In [57]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

def make_l1_selection_pipeline(classifier):

    # First: handle missing values and constant features
    preprocessing = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("variance_filter", VarianceThreshold()),
        ("scaler", StandardScaler())
    ])

    return Pipeline([
        ("preprocessing", preprocessing),

        ("feature_selection", SelectFromModel(
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=0.3,
                max_iter=3000,
                random_state=42
            ),
            threshold=1e-8
        )),

        ("classifier", classifier)
    ])

In [58]:
l1_rf = make_l1_selection_pipeline(
    RandomForestClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=2
    )
)

l1_l2 = make_l1_selection_pipeline(
    LogisticRegression(
        penalty="l2",
        C=1.0,
        max_iter=3000,
        random_state=42
    )
)

In [59]:
from sklearn.base import clone

test_pipeline = clone(l1_rf)

test_pipeline.fit(
    X_full_extended[train_idx],
    y_train
)

selector = test_pipeline.named_steps["feature_selection"]

print(
    "Number of selected features:",
    selector.get_support().sum()
)

print(
    "Number of features before selection:",
    len(selector.get_support())
)

Number of selected features: 1523
Number of features before selection: 2189


In [60]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

def make_comparison_pipeline(classifier, use_l1=False):

    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance_filter", VarianceThreshold()),
        ("scaler", StandardScaler())
    ]

    if use_l1:
        steps.append((
            "feature_selection",
            SelectFromModel(
                LogisticRegression(
                    penalty="l1",
                    solver="liblinear",
                    C=0.3,
                    max_iter=3000,
                    random_state=42
                ),
                threshold=1e-8
            )
        ))

    steps.append(("classifier", classifier))

    return Pipeline(steps)


rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

lr_l2 = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=3000,
    random_state=42
)

selection_models = {
    "RF - Full": make_comparison_pipeline(clone(rf), False),
    "RF - L1 Selected": make_comparison_pipeline(clone(rf), True),
    "LR L2 - Full": make_comparison_pipeline(clone(lr_l2), False),
    "LR L2 - L1 Selected": make_comparison_pipeline(clone(lr_l2), True)
}

In [61]:
from sklearn.model_selection import GroupKFold
import pandas as pd
import time

cv = GroupKFold(n_splits=5)

X = X_full_extended[train_idx]

selection_results = []

for name, template in selection_models.items():

    print("Evaluating:", name)

    for fold, (tr, va) in enumerate(
        cv.split(X, y_train, groups=groups_train),
        start=1
    ):

        model = clone(template)

        start = time.perf_counter()
        model.fit(X[tr], y_train[tr])
        train_time = time.perf_counter() - start

        probabilities = model.predict_proba(X[va])[:, 1]

        metrics = evaluate_model(
            y_true=y_train[va],
            probabilities=probabilities,
            model_name=name,
            threshold=0.5
        )

        if "feature_selection" in model.named_steps:
            selected_count = int(
                model.named_steps["feature_selection"]
                .get_support().sum()
            )
        else:
            selected_count = int(
                model.named_steps["variance_filter"]
                .get_support().sum()
            )

        metrics["Fold"] = fold
        metrics["Selected_Features"] = selected_count
        metrics["Train_Time_s"] = train_time

        selection_results.append(metrics)

selection_cv_df = pd.DataFrame(selection_results)

Evaluating: RF - Full
Evaluating: RF - L1 Selected
Evaluating: LR L2 - Full
Evaluating: LR L2 - L1 Selected


In [62]:
columns = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Selected_Features",
    "Train_Time_s"
]

selection_summary = (
    selection_cv_df
    .groupby("Model")[columns]
    .agg(["mean", "std"])
    .sort_values(("AUROC", "mean"), ascending=False)
)

display(selection_summary.round(4))

AUROC           AUPRC         Accuracy          \
                       mean     std    mean     std     mean     std   
Model                                                                  
RF - L1 Selected     0.8678  0.0253  0.8764  0.0378   0.7870  0.0290   
RF - Full            0.8676  0.0291  0.8772  0.0370   0.7882  0.0337   
LR L2 - L1 Selected  0.7785  0.0095  0.7617  0.0290   0.7344  0.0168   
LR L2 - Full         0.7663  0.0108  0.7492  0.0309   0.7226  0.0137   

                    Sensitivity         Specificity         Precision          \
                           mean     std        mean     std      mean     std   
Model                                                                           
RF - L1 Selected         0.7939  0.0278      0.7793  0.0348    0.7987  0.0405   
RF - Full                0.7853  0.0302      0.7910  0.0461    0.8060  0.0478   
LR L2 - L1 Selected      0.7518  0.0297      0.7147  0.0109    0.7439  0.0312   
LR L2 - Full             0.7433  0.0299      0.6995  0.0092    0.7319  0.0281   

                         F1         Selected_Features          Train_Time_s  \
                       mean     std              mean      std         mean   
Model                                                                         
RF - L1 Selected     0.7961  0.0323            1378.8  15.5467      13.1285   
RF - Full            0.7952  0.0367            2173.2   4.6583       4.0740   
LR L2 - L1 Selected  0.7476  0.0274            1378.8  15.5467      11.3260   
LR L2 - Full         0.7373  0.0237            2173.2   4.6583       1.8420   

                             
                        std  
Model                        
RF - L1 Selected     3.4935  
RF - Full            0.4465  
LR L2 - L1 Selected  2.8973  
LR L2 - Full         0.1289

In [63]:
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
import pandas as pd
import time

tree_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=2
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=2
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.1,
        max_leaf_nodes=31,
        random_state=42
    )
}

In [64]:
X = fp_type_sets["Feature_Count"][train_idx]

cv = GroupKFold(n_splits=5)

tree_results = []

for name, template in tree_models.items():

    print("Evaluating:", name)

    for fold, (tr, va) in enumerate(
        cv.split(X, y_train, groups=groups_train),
        start=1
    ):

        model = clone(template)

        start = time.perf_counter()
        model.fit(X[tr], y_train[tr])
        train_time = time.perf_counter() - start

        start = time.perf_counter()
        probabilities = model.predict_proba(X[va])[:, 1]
        inference_time = time.perf_counter() - start

        result = evaluate_model(
            y_true=y_train[va],
            probabilities=probabilities,
            model_name=name,
            threshold=0.5
        )

        result["Fold"] = fold
        result["Train_Time_s"] = train_time
        result["Inference_ms_per_molecule"] = (
            inference_time * 1000 / len(va)
        )

        tree_results.append(result)

tree_cv_df = pd.DataFrame(tree_results)

Evaluating: Random Forest
Evaluating: Extra Trees
Evaluating: HistGradientBoosting


In [65]:
metrics = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Train_Time_s",
    "Inference_ms_per_molecule"
]

tree_summary = (
    tree_cv_df
    .groupby("Model")[metrics]
    .agg(["mean", "std"])
    .sort_values(("AUROC", "mean"), ascending=False)
)

display(tree_summary.round(4))

AUROC           AUPRC         Accuracy          \
                        mean     std    mean     std     mean     std   
Model                                                                   
Random Forest         0.8688  0.0282  0.8804  0.0370   0.7879  0.0307   
Extra Trees           0.8673  0.0277  0.8790  0.0392   0.7822  0.0303   
HistGradientBoosting  0.8600  0.0295  0.8701  0.0421   0.7791  0.0332   

                     Sensitivity         Specificity         Precision  \
                            mean     std        mean     std      mean   
Model                                                                    
Random Forest             0.7867  0.0274      0.7884  0.0405    0.8044   
Extra Trees               0.7848  0.0297      0.7787  0.0413    0.7969   
HistGradientBoosting      0.7806  0.0266      0.7776  0.0436    0.7945   

                                  F1         Train_Time_s          \
                         std    mean     std         mean     std   
Model                                                               
Random Forest         0.0427  0.7953  0.0340       1.7906  0.3439   
Extra Trees           0.0435  0.7906  0.0339       3.4965  1.1126   
HistGradientBoosting  0.0488  0.7873  0.0366       4.5905  1.3789   

                     Inference_ms_per_molecule          
                                          mean     std  
Model                                                   
Random Forest                           0.0346  0.0066  
Extra Trees                             0.0383  0.0063  
HistGradientBoosting                    0.0165  0.0056

In [66]:
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.model_selection import (
    GroupKFold,
    RandomizedSearchCV
)

from sklearn.base import clone
import numpy as np
import pandas as pd
import time

model_configs = {
    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=42,
            n_jobs=2
        ),
        "params": {
            "n_estimators": [100, 200, 400],
            "max_depth": [None, 15, 30],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", 0.2, 0.5],
            "min_samples_split": [2, 5]
        }
    },

    "Extra Trees": {
        "model": ExtraTreesClassifier(
            random_state=42,
            n_jobs=2
        ),
        "params": {
            "n_estimators": [100, 200, 400],
            "max_depth": [None, 15, 30],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", 0.2, 0.5],
            "min_samples_split": [2, 5]
        }
    },

    "HistGradientBoosting": {
        "model": HistGradientBoostingClassifier(
            random_state=42,
            early_stopping=False
        ),
        "params": {
            "max_iter": [100, 150, 250],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_leaf_nodes": [15, 31, 63],
            "min_samples_leaf": [10, 20, 40],
            "l2_regularization": [0.0, 0.1, 1.0]
        }
    }
}

In [67]:
from sklearn.metrics import make_scorer, recall_score

specificity_scorer = make_scorer(
    recall_score,
    pos_label=0
)

X = fp_type_sets["Feature_Count"][train_idx]
y_cv = y_train
groups = groups_train

outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=3)

nested_results = []
best_params_results = []

for model_name, config in model_configs.items():

    print(f"\nRunning: {model_name}")

    for fold, (tr, va) in enumerate(
        outer_cv.split(X, y_cv, groups=groups),
        start=1
    ):

        search = RandomizedSearchCV(
            estimator=clone(config["model"]),
            param_distributions=config["params"],
            n_iter=6,
            scoring={
                "AUROC": "roc_auc",
                "AUPRC": "average_precision",
                "Sensitivity": "recall",
                "Precision": "precision",
                "Specificity": specificity_scorer
            },
            refit="AUROC",
            cv=inner_cv.split(
                X[tr],
                y_cv[tr],
                groups=groups[tr]
            ),
            random_state=42,
            n_jobs=1,
            error_score="raise"
        )

        start = time.perf_counter()

        search.fit(X[tr], y_cv[tr])

        train_time = time.perf_counter() - start

        best_model = search.best_estimator_

        start = time.perf_counter()

        probabilities = best_model.predict_proba(
            X[va]
        )[:, 1]

        inference_time = time.perf_counter() - start

        metrics = evaluate_model(
            y_true=y_cv[va],
            probabilities=probabilities,
            model_name=model_name,
            threshold=0.5
        )

        metrics["Fold"] = fold
        metrics["Train_Time_s"] = train_time
        metrics["Inference_ms_per_molecule"] = (
            inference_time * 1000 / len(va)
        )

        nested_results.append(metrics)

        best_params_results.append({
            "Model": model_name,
            "Fold": fold,
            "Best_Params": search.best_params_,
            "Inner_AUROC": search.best_score_
        })

        print(
            f" Fold {fold}: "
            f"AUROC={metrics['AUROC']:.4f}, "
            f"Sens={metrics['Sensitivity']:.4f}, "
            f"Spec={metrics['Specificity']:.4f}"
        )

nested_all_df = pd.DataFrame(nested_results)

best_params_df = pd.DataFrame(best_params_results)


Running: Random Forest
 Fold 1: AUROC=0.9008, Sens=0.8166, Spec=0.8183
 Fold 2: AUROC=0.8955, Sens=0.8335, Spec=0.7762
 Fold 3: AUROC=0.8472, Sens=0.7625, Spec=0.7459
 Fold 4: AUROC=0.8301, Sens=0.7674, Spec=0.7456
 Fold 5: AUROC=0.8937, Sens=0.8121, Spec=0.8554

Running: Extra Trees
 Fold 1: AUROC=0.9008, Sens=0.8166, Spec=0.8230
 Fold 2: AUROC=0.8995, Sens=0.8247, Spec=0.8003
 Fold 3: AUROC=0.8550, Sens=0.7883, Spec=0.7608
 Fold 4: AUROC=0.8350, Sens=0.7515, Spec=0.7441
 Fold 5: AUROC=0.8956, Sens=0.7891, Spec=0.8539

Running: HistGradientBoosting
 Fold 1: AUROC=0.8945, Sens=0.8331, Spec=0.8214
 Fold 2: AUROC=0.8967, Sens=0.8310, Spec=0.8021
 Fold 3: AUROC=0.8459, Sens=0.7754, Spec=0.7668
 Fold 4: AUROC=0.8321, Sens=0.7602, Spec=0.7603
 Fold 5: AUROC=0.8811, Sens=0.7877, Spec=0.8376


In [68]:
metrics_to_report = [
    "AUROC",
    "AUPRC",
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Train_Time_s",
    "Inference_ms_per_molecule"
]

nested_summary = (
    nested_all_df
    .groupby("Model")[metrics_to_report]
    .agg(["mean", "std"])
    .sort_values(
        ("AUROC", "mean"),
        ascending=False
    )
)

display(nested_summary.round(4))

AUROC           AUPRC         Accuracy          \
                        mean     std    mean     std     mean     std   
Model                                                                   
Extra Trees           0.8772  0.0303  0.8883  0.0386   0.7955  0.0327   
Random Forest         0.8735  0.0325  0.8835  0.0420   0.7942  0.0364   
HistGradientBoosting  0.8701  0.0294  0.8777  0.0401   0.7980  0.0302   

                     Sensitivity         Specificity         Precision  \
                            mean     std        mean     std      mean   
Model                                                                    
Extra Trees               0.7940  0.0288      0.7964  0.0448    0.8118   
Random Forest             0.7984  0.0316      0.7883  0.0479    0.8069   
HistGradientBoosting      0.7975  0.0330      0.7976  0.0336    0.8128   

                                  F1         Train_Time_s           \
                         std    mean     std         mean      std   
Model                                                                
Extra Trees           0.0470  0.8026  0.0361     240.0497  10.7837   
Random Forest         0.0474  0.8025  0.0387     138.3284   8.2914   
HistGradientBoosting  0.0403  0.8049  0.0355     177.5006  15.2450   

                     Inference_ms_per_molecule          
                                          mean     std  
Model                                                   
Extra Trees                             0.0734  0.0295  
Random Forest                           0.0321  0.0077  
HistGradientBoosting                    0.0217  0.0168

In [69]:
params_expanded = pd.json_normalize(
    best_params_df["Best_Params"]
)

params_table = pd.concat(
    [
        best_params_df[
            ["Model", "Fold", "Inner_AUROC"]
        ].reset_index(drop=True),
        params_expanded.reset_index(drop=True)
    ],
    axis=1
)

display(
    params_table.sort_values(
        ["Model", "Fold"]
    ).round(4)
)

,Model,Fold,Inner_AUROC,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,max_leaf_nodes,max_iter,learning_rate,l2_regularization
5,Extra Trees,1,0.8549,400.0,5.0,2,0.2,NaN,NaN,NaN,NaN,NaN
6,Extra Trees,2,0.8553,400.0,5.0,2,0.2,NaN,NaN,NaN,NaN,NaN
7,Extra Trees,3,0.8570,400.0,5.0,1,0.2,30.0,NaN,NaN,NaN,NaN
8,Extra Trees,4,0.8657,400.0,5.0,2,0.2,NaN,NaN,NaN,NaN,NaN
9,Extra Trees,5,0.8484,400.0,5.0,2,0.2,NaN,NaN,NaN,NaN,NaN
10,HistGradientBoosting,1,0.8486,NaN,NaN,20,NaN,NaN,63.0,150.0,0.10,1.0
11,HistGradientBoosting,2,0.8452,NaN,NaN,20,NaN,NaN,63.0,150.0,0.10,1.0
12,HistGradientBoosting,3,0.8477,NaN,NaN,20,NaN,NaN,63.0,150.0,0.10,1.0
13,HistGradientBoosting,4,0.8621,NaN,NaN,10,NaN,NaN,63.0,250.0,0.03,0.0
14,HistGradientBoosting,5,0.8413,NaN,NaN,40,NaN,NaN,63.0,150.0,0.10,0.1


In [70]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, recall_score
import pandas as pd

specificity_scorer = make_scorer(recall_score, pos_label=0)

search_configs = {
    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=42, n_jobs=2
        ),
        "params": {
            "n_estimators": [200, 400, 600],
            "max_depth": [20, 30, None],
            "min_samples_leaf": [1, 2],
            "max_features": ["sqrt", 0.1, 0.2]
        }
    },
    "Extra Trees": {
        "model": ExtraTreesClassifier(
            random_state=42, n_jobs=2
        ),
        "params": {
            "n_estimators": [200, 400, 600],
            "max_depth": [20, 30, None],
            "min_samples_leaf": [1, 2],
            "max_features": ["sqrt", 0.1, 0.2]
        }
    }
}

In [71]:
X = fp_type_sets["Feature_Count"][train_idx]
y_cv = y_train
groups = groups_train

cv = GroupKFold(n_splits=5)

search_results = []
fitted_searches = {}

for name, config in search_configs.items():

    print("Searching:", name)

    search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=20,
        scoring={
            "AUROC": "roc_auc",
            "AUPRC": "average_precision",
            "Sensitivity": "recall",
            "Specificity": specificity_scorer,
            "Precision": "precision",
            "F1": "f1",
            "Accuracy": "accuracy"
        },
        refit="AUROC",
        cv=cv,
        random_state=42,
        n_jobs=1,
        error_score="raise"
    )

    search.fit(X, y_cv, groups=groups)

    fitted_searches[name] = search

    result = pd.DataFrame(search.cv_results_)
    result["Model"] = name

    search_results.append(result)

    print("Best AUROC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

search_df = pd.concat(search_results, ignore_index=True)

Searching: Random Forest
Best AUROC: 0.8776
Best parameters: {'n_estimators': 400, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Searching: Extra Trees
Best AUROC: 0.8769
Best parameters: {'n_estimators': 400, 'min_samples_leaf': 2, 'max_features': 0.2, 'max_depth': None}


In [72]:
columns = [
    "Model",
    "params",
    "mean_test_AUROC",
    "std_test_AUROC",
    "mean_test_AUPRC",
    "mean_test_Sensitivity",
    "mean_test_Specificity",
    "mean_test_Precision",
    "mean_test_F1",
    "mean_test_Accuracy",
    "mean_fit_time"
]

search_table = search_df[columns].sort_values(
    "mean_test_AUROC",
    ascending=False
)

display(search_table.round(4).head(20))

,Model,params,mean_test_AUROC,std_test_AUROC,mean_test_AUPRC,mean_test_Sensitivity,mean_test_Specificity,mean_test_Precision,mean_test_F1,mean_test_Accuracy,mean_fit_time
16,Random Forest,"{'n_estimators': 400, 'min_samples_leaf': 1, '...",0.8776,0.0276,0.8878,0.7986,0.7917,0.8093,0.8038,0.7958,3.4740
27,Extra Trees,"{'n_estimators': 400, 'min_samples_leaf': 2, '...",0.8769,0.0271,0.8879,0.7894,0.7945,0.8096,0.7992,0.7924,25.4667
39,Extra Trees,"{'n_estimators': 200, 'min_samples_leaf': 2, '...",0.8755,0.0273,0.8852,0.7878,0.7988,0.8125,0.7999,0.7936,13.1322
34,Extra Trees,"{'n_estimators': 400, 'min_samples_leaf': 2, '...",0.8755,0.0276,0.8855,0.7874,0.7970,0.8110,0.7989,0.7926,24.6244
17,Random Forest,"{'n_estimators': 200, 'min_samples_leaf': 1, '...",0.8750,0.0276,0.8862,0.7951,0.7957,0.8116,0.8031,0.7959,4.1865
12,Random Forest,"{'n_estimators': 600, 'min_samples_leaf': 1, '...",0.8750,0.0283,0.8859,0.7952,0.7943,0.8102,0.8025,0.7951,12.3443
4,Random Forest,"{'n_estimators': 600, 'min_samples_leaf': 1, '...",0.8749,0.0291,0.8857,0.7974,0.7911,0.8082,0.8026,0.7949,15.5159
38,Extra Trees,"{'n_estimators': 200, 'min_samples_leaf': 2, '...",0.8748,0.0272,0.8861,0.7916,0.7889,0.8054,0.7983,0.7908,7.0908
29,Extra Trees,"{'n_estimators': 600, 'min_samples_leaf': 1, '...",0.8747,0.0303,0.8828,0.7953,0.7883,0.8062,0.8005,0.7924,39.8530
21,Extra Trees,"{'n_estimators': 400, 'min_samples_leaf': 1, '...",0.8745,0.0312,0.8813,0.7970,0.7882,0.8063,0.8014,0.7932,28.4251


In [73]:
for name, search in fitted_searches.items():

    print("\nModel:", name)
    print("Best CV AUROC:", round(search.best_score_, 4))

    print("Best parameters:")

    for parameter, value in search.best_params_.items():
        print(f"  {parameter}: {value}")


Model: Random Forest
Best CV AUROC: 0.8776
Best parameters:
  n_estimators: 400
  min_samples_leaf: 1
  max_features: sqrt
  max_depth: None

Model: Extra Trees
Best CV AUROC: 0.8769
Best parameters:
  n_estimators: 400
  min_samples_leaf: 2
  max_features: 0.2
  max_depth: None


In [74]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

X = fp_type_sets["Feature_Count"][train_idx]
y_cv = y_train
groups = groups_train

cv = GroupKFold(n_splits=5)

oof_predictions = {}

for name, search in fitted_searches.items():

    model = clone(search.best_estimator_)

    probabilities = cross_val_predict(
        model,
        X,
        y_cv,
        groups=groups,
        cv=cv,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    oof_predictions[name] = probabilities

    print(
        name,
        "OOF AUROC:",
        round(roc_auc_score(y_cv, probabilities), 4),
        "OOF AUPRC:",
        round(average_precision_score(y_cv, probabilities), 4)
    )

Random Forest OOF AUROC: 0.8785 OOF AUPRC: 0.8901
Extra Trees OOF AUROC: 0.8779 OOF AUPRC: 0.8903


In [75]:
from sklearn.metrics import confusion_matrix

def optimize_thresholds(y_true, probabilities, margin=0.03):

    thresholds = np.unique(
        np.concatenate(([0.0], probabilities, [1.0]))
    )

    rows = []

    for t in thresholds:
        pred = (probabilities >= t).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, pred, labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if tp + fn else 0
        specificity = tn / (tn + fp) if tn + fp else 0
        precision = tp / (tp + fp) if tp + fp else 0

        rows.append({
            "Threshold": t,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision
        })

    table = pd.DataFrame(rows)

    minimum = 0.70 + margin

    objectives = {
        "Sensitivity": ("Sensitivity", "Specificity"),
        "Specificity": ("Specificity", "Sensitivity"),
        "Precision": ("Precision", "Sensitivity")
    }

    selected = []

    for goal, (metric, constraint) in objectives.items():

        eligible = table[table[constraint] > minimum]

        if eligible.empty:
            continue

        best = eligible.sort_values(
            [metric, constraint],
            ascending=[False, False]
        ).iloc[0]

        selected.append({
            "Goal": goal,
            **best.to_dict()
        })

    return pd.DataFrame(selected)

In [76]:
threshold_results = []

for name, probabilities in oof_predictions.items():

    result = optimize_thresholds(
        y_true=y_cv,
        probabilities=probabilities,
        margin=0.03
    )

    result["Model"] = name

    threshold_results.append(result)

threshold_summary = pd.concat(
    threshold_results,
    ignore_index=True
)

display(
    threshold_summary[
        ["Model", "Goal", "Threshold",
         "Sensitivity", "Specificity", "Precision"]
    ].round(4)
)

,Model,Goal,Threshold,Sensitivity,Specificity,Precision
0,Random Forest,Sensitivity,0.4458,0.8562,0.7304,0.7787
1,Random Forest,Specificity,0.5600,0.7308,0.8515,0.8450
2,Random Forest,Precision,0.5600,0.7308,0.8515,0.8450
3,Extra Trees,Sensitivity,0.4326,0.8529,0.7304,0.7780
4,Extra Trees,Specificity,0.5732,0.7305,0.8555,0.8485
5,Extra Trees,Precision,0.5732,0.7305,0.8555,0.8485


In [77]:
import numpy as np
import pandas as pd

thresholds = np.round(
    np.arange(0.44, 0.581, 0.01), 2
)

threshold_results = []

for model_name, probabilities in oof_predictions.items():

    for threshold in thresholds:

        metrics = evaluate_model(
            y_true=y_train,
            probabilities=probabilities,
            model_name=model_name,
            threshold=threshold
        )

        threshold_results.append(metrics)

threshold_grid_df = pd.DataFrame(threshold_results)

display(
    threshold_grid_df[
        [
            "Model",
            "Threshold",
            "Sensitivity",
            "Specificity",
            "Precision",
            "F1",
            "Accuracy"
        ]
    ].round(4)
)

,Model,Threshold,Sensitivity,Specificity,Precision,F1,Accuracy
0,Random Forest,0.44,0.8632,0.7205,0.7739,0.8161,0.7955
1,Random Forest,0.45,0.8532,0.7347,0.7809,0.8154,0.7970
2,Random Forest,0.46,0.8434,0.7470,0.7870,0.8142,0.7977
3,Random Forest,0.47,0.8334,0.7587,0.7929,0.8126,0.7980
4,Random Forest,0.48,0.8245,0.7692,0.7983,0.8112,0.7983
5,Random Forest,0.49,0.8142,0.7790,0.8033,0.8087,0.7975
6,Random Forest,0.50,0.8020,0.7892,0.8083,0.8051,0.7959
7,Random Forest,0.51,0.7903,0.8000,0.8141,0.8020,0.7949
8,Random Forest,0.52,0.7784,0.8102,0.8196,0.7985,0.7935
9,Random Forest,0.53,0.7681,0.8225,0.8274,0.7967,0.7939


In [78]:
comparison = threshold_grid_df.pivot(
    index="Threshold",
    columns="Model",
    values=[
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1"
    ]
)

display(comparison.round(4))

Sensitivity               Specificity                 Precision  \
Model     Extra Trees Random Forest Extra Trees Random Forest Extra Trees   
Threshold                                                                   
0.44           0.8459        0.8632      0.7362        0.7205      0.7804   
0.45           0.8382        0.8532      0.7473        0.7347      0.7861   
0.46           0.8256        0.8434      0.7612        0.7470      0.7930   
0.47           0.8173        0.8334      0.7698        0.7587      0.7973   
0.48           0.8103        0.8245      0.7787        0.7692      0.8023   
0.49           0.8014        0.8142      0.7880        0.7790      0.8073   
0.50           0.7906        0.8020      0.7945        0.7892      0.8100   
0.51           0.7848        0.7903      0.8046        0.8000      0.8166   
0.52           0.7781        0.7784      0.8136        0.8102      0.8222   
0.53           0.7689        0.7681      0.8210        0.8225      0.8264   
0.54           0.7595        0.7539      0.8314        0.8330      0.8331   
0.55           0.7492        0.7411      0.8404        0.8431      0.8387   
0.56           0.7405        0.7308      0.8465        0.8515      0.8425   
0.57           0.7330        0.7202      0.8536        0.8604      0.8473   
0.58           0.7247        0.7100      0.8613        0.8669      0.8527   

                                 F1                
Model     Random Forest Extra Trees Random Forest  
Threshold                                          
0.44             0.7739      0.8118        0.8161  
0.45             0.7809      0.8113        0.8154  
0.46             0.7870      0.8090        0.8142  
0.47             0.7929      0.8072        0.8126  
0.48             0.7983      0.8063        0.8112  
0.49             0.8033      0.8044        0.8087  
0.50             0.8083      0.8002        0.8051  
0.51             0.8141      0.8003        0.8020  
0.52             0.8196      0.7995        0.7985  
0.53             0.8274      0.7966        0.7967  
0.54             0.8334      0.7946        0.7916  
0.55             0.8396      0.7914        0.7873  
0.56             0.8450      0.7882        0.7838  
0.57             0.8511      0.7860        0.7802  
0.58             0.8553      0.7835        0.7759

In [79]:
balanced_results = threshold_grid_df.copy()

balanced_results["Balanced_Accuracy"] = (
    balanced_results["Sensitivity"]
    + balanced_results["Specificity"]
) / 2

eligible = balanced_results[
    (balanced_results["Sensitivity"] > 0.73)
    &
    (balanced_results["Specificity"] > 0.73)
]

best_balanced = (
    eligible.sort_values(
        "Balanced_Accuracy",
        ascending=False
    )
    .groupby("Model")
    .head(1)
)

display(
    best_balanced[
        [
            "Model",
            "Threshold",
            "Balanced_Accuracy",
            "Sensitivity",
            "Specificity",
            "Precision",
            "F1"
        ]
    ].round(4)
)

,Model,Threshold,Balanced_Accuracy,Sensitivity,Specificity,Precision,F1
4,Random Forest,0.48,0.7969,0.8245,0.7692,0.7983,0.8112
23,Extra Trees,0.52,0.7958,0.7781,0.8136,0.8222,0.7995
